In [ ]:
# =================================================================
# STEP 1: Install Requirements
# =================================================================
!pip install transformers huggingface_hub -q

In [ ]:
# =================================================================
# STEP 2: Import Libraries & Setup Directory
# =================================================================
import os
import shutil
from transformers import AutoTokenizer, AutoModel, AutoConfig
from huggingface_hub import snapshot_download

In [ ]:
DATASET_DIR = "nexus_absa_assets"
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR)

# ساخت زیرپوشه‌ها
os.makedirs(f"{DATASET_DIR}/models", exist_ok=True)
os.makedirs(f"{DATASET_DIR}/cache", exist_ok=True)

In [ ]:
# =================================================================
# STEP 3: Download Models (The Heavy Part)
# =================================================================
MODEL_IDS = [
    "FacebookAI/roberta-base",
    "microsoft/deberta-v3-base",
    "Qwen/Qwen2-0.5B"
]

print("--- Starting Model Downloads ---")
for model_id in MODEL_IDS:
    model_name = model_id.split("/")[-1]
    print(f"Downloading {model_name}...")
    
    save_path = f"{DATASET_DIR}/models/{model_name}"
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.save_pretrained(save_path)
    
    model = AutoModel.from_pretrained(model_id, trust_remote_code=True)
    model.save_pretrained(save_path)
    
    print(f"Successfully saved {model_name} to {save_path}")

In [ ]:
# =================================================================
# STEP 4: Move Caches (Generated in previous step)
# =================================================================
CACHE_FILES = [
    "absa_conceptnet_cache.json",
    "absa_senticnet_cache.json"
]

print("\n--- Moving Cache Files ---")
for cache_file in CACHE_FILES:
    if os.path.exists(cache_file):
        shutil.copy(cache_file, f"{DATASET_DIR}/cache/{cache_file}")
        print(f"Copied {cache_file} to dataset folder.")
    else:
        print(f"WARNING: {cache_file} not found! Run build_absa_caches.py first.")

In [ ]:
# =================================================================
# STEP 5: Create Dataset Metadata & Zip
# =================================================================
print("\n--- Creating Final Archive ---")
# فشرده‌سازی پوشه برای آپلود در کگل
shutil.make_archive("nexus_absa_kaggle_input", 'zip', DATASET_DIR)

print("\n" + "="*50)
print("SUCCESS: 'nexus_absa_kaggle_input.zip' is ready!")
print("Action: Download this file and upload it to Kaggle as a Private Dataset.")
print("="*50)